<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This notebook introduces students of computational linguistics to the `udapi-python` library, a powerful tool for working with Universal Dependencies treebanks. We will cover installation, downloading a treebank, and performing a basic search for linguistic features.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [2]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 10.4 MB/s eta 0:00:00
usage: udapy [optional_arguments] scenario

udapy - Python interface to Udapi - API for Universal Dependencies

Examples of usage:
  udapy -s read.Sentences udpipe.En < in.txt > out.conllu
  udapy -T < sample.conllu | less -R
  udapy -HAM ud.MarkBugs < sample.conllu > bugs.html

positional arguments:
  scenario              A sequence of blocks and their parameters.

options:
  -h, --help            show this help message and exit
  -q, --quiet           Warning, info and debug messages are suppressed. Only fatal errors are reported.
  -v, --verbose         Warning, info and debug messages are printed to the STDERR.
  -s, --save            Add write.Conllu to the end of the scenario
  -T, --save_text_mode_trees
                        Add write.TextModeTrees color=1 to the end of the scenario
  -H, --save_html       Add write.TextModeTreesHtml color=1 to the end of the scenario
  -A, --save_all_attributes


## 2. Download and Load a UD Treebank

Udapi allows easy access to Universal Dependencies treebanks. As an example, we will download several treebanks from the parallel UD collection. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks.

In [41]:
%%bash
rm -rf UD_*
#git clone https://github.com/UniversalDependencies/UD_Czech-PUD.git
#git clone https://github.com/UniversalDependencies/UD_English-PUD.git
#git clone https://github.com/UniversalDependencies/UD_Spanish-PUD.git
#git clone https://github.com/UniversalDependencies/UD_Portuguese-PUD.git
git clone https://github.com/UniversalDependencies/UD_Czech-FicTree.git
git clone https://github.com/UniversalDependencies/UD_Portuguese-Porttinari.git

Cloning into 'UD_Czech-FicTree'...
Cloning into 'UD_Portuguese-Porttinari'...


Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

In [43]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U file.
# TODO: Change this variable to explore different datasets.
treebank = 'UD_Czech-FicTree' # Student can modify this line.

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file. That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")


Found 3 CoNLL-U files in UD_Czech-FicTree:
UD_Czech-FicTree/cs_fictree-ud-dev.conllu
UD_Czech-FicTree/cs_fictree-ud-test.conllu
UD_Czech-FicTree/cs_fictree-ud-train.conllu
Loading UD_Czech-FicTree...
Loaded 12760 sentences from UD_Czech-FicTree.


## 3. Find and Print Lemmas of Personal Pronouns

Now, let's explore the loaded treebank. We'll iterate through each word (node) in every sentence (bundle) in every CoNLL-U file of the treebank and identify personal pronouns (UPOS tag `PRON` and PronType feature `Prs`). Then, we'll print their lemmas.

In [44]:
personal_pronoun_lemmas = set()

for doc in filedocs:
    # Bundle is a unit that corresponds to one sentence in Udapi.
    # In theory it may have multiple trees for that sentence; but normally there is just one.
    for bundle in doc.bundles:
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            # Check if the word is a pronoun (UPOS tag 'PRON')
            # and if its PronType feature is 'Prs' (Personal Pronoun)
            if node.upos == 'PRON' and node.feats['PronType'] == 'Prs':
                personal_pronoun_lemmas.add(node.lemma)

print(f"\nLemmas of personal pronouns found in {treebank}:")
for lemma in sorted(list(personal_pronoun_lemmas)):
    print(lemma)


Lemmas of personal pronouns found in UD_Czech-FicTree:
já
my
on
se
ty
von


## 4. Collect All Forms of All Lemmas

This will help us to use various criteria to select and display paradigms of individual lexemes.

In [52]:
from collections import defaultdict

# dictionary will store: {UPOS: {lemma: {form: {featstring: count}}}}
dictionary = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int))))
all_lemmas = {}
all_forms = {}

for doc in filedocs: # Iterate through each Document in the list of loaded CoNLL-U files
    for bundle in doc.bundles: # Iterate through bundles within each Document
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            # Exclude nodes marked as typos
            if not (node.feats and node.feats.get('Typo') == 'Yes'):
                upos = node.upos
                lemma = node.lemma.lower()
                form = node.form.lower()
                # The feature ExtPos, if present, is not interesting.
                # It is not about morphology but about syntax of the sentence.
                # Note: This will erase the feature from the loaded document, so we will not be able to work with it. It will however not destroy it in the file on the disk.
                node.feats['ExtPos'] = ''
                featstring = str(node.feats) if node.feats else 'NO_FEATS'
                dictionary[upos][lemma][form][featstring] += 1
                all_lemmas[lemma] = True
                all_forms[form] = True

print(f"Collected {len(all_forms)} distinct forms and {len(all_lemmas)} distinct lemmas in total.")
print(f"This makes the average morphological richness {len(all_forms)/len(all_lemmas)} forms per lemma.")

Collected 27084 distinct forms and 13723 distinct lemmas in total.
This makes the average morphological richness 1.9736209283684325 forms per lemma.


In [65]:
# @title Select the paradigm with most forms and print it.
import pandas as pd

maxforms = 0
winners = []
for upos in dictionary:
    uposdict = dictionary[upos]
    for lemma in uposdict:
        lemmadict = uposdict[lemma]
        nforms = len(lemmadict)
        # We can choose to consider only one UPOS category.
        if upos != 'NOUN':
            continue
        if nforms > maxforms:
            maxforms = nforms
            winners = [(upos, lemma)]
        elif nforms == maxforms:
            winners.append((upos, lemma))
print(f"Maximum number of forms found in one lexeme is {maxforms}.")
print(f"It was observed with the following UPOS + LEMMA combinations:")
for upos, lemma in winners:
    print(f"{upos} {lemma}")

# Change this to == 1 if you only want to see unique winners.
if len(winners) >= 1:
    print()
    selected_upos = winners[0][0]
    selected_lemma = winners[0][1]
    lemmadict = dictionary[selected_upos][selected_lemma]

    # Prepare data for a DataFrame
    table_data = []
    for form in sorted(lemmadict):
        for featstring in sorted(lemmadict[form]):
            table_data.append({
                'Form': form,
                'Features': featstring,
                'Count': lemmadict[form][featstring]
            })

    df_paradigm = pd.DataFrame(table_data)
    print(f"Paradigm for {selected_upos} '{selected_lemma}':")

    # Manually format for left-aligned first two columns and right-aligned last column
    if not df_paradigm.empty:
        # Calculate max widths for 'Form' and 'Features' columns
        max_form_width = max(len(str(x)) for x in df_paradigm['Form'].tolist() + ['Form'])
        max_features_width = max(len(str(x)) for x in df_paradigm['Features'].tolist() + ['Features'])
        max_count_width = max(len(str(x)) for x in df_paradigm['Count'].tolist() + ['Count'])

        # Print header
        print(f"{('Form').ljust(max_form_width)} {('Features').ljust(max_features_width)} {('Count').rjust(max_count_width)}")
        # Print separator
        print(f"{'-'*max_form_width} {'-'*max_features_width} {'-'*max_count_width}")

        # Print data rows
        for index, row in df_paradigm.iterrows():
            print(f"{str(row['Form']).ljust(max_form_width)} {str(row['Features']).ljust(max_features_width)} {str(row['Count']).rjust(max_count_width)}")


Maximum number of forms found in one lexeme is 13.
It was observed with the following UPOS + LEMMA combinations:
NOUN rok

Paradigm for NOUN 'rok':
Form   Features                                      Count
------ --------------------------------------------- -----
let    Case=Gen|Gender=Neut|Number=Plur                 81
leta   Case=Acc|Gender=Neut|Number=Plur|Style=Coll       1
letech Case=Loc|Gender=Neut|Number=Plur                 22
lety   Case=Ins|Gender=Neut|Number=Plur                  6
léta   Case=Acc|Gender=Neut|Number=Plur                 27
léta   Case=Gen|Gender=Neut|Number=Sing                  1
léta   Case=Nom|Gender=Neut|Number=Plur                  2
létech Case=Loc|Gender=Neut|Number=Plur                  1
léty   Case=Ins|Gender=Neut|Number=Plur                  2
roce   Animacy=Inan|Case=Loc|Gender=Masc|Number=Sing    19
rok    Animacy=Inan|Case=Acc|Gender=Masc|Number=Sing    30
rok    Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing     5
roka   Animacy=Inan|Case=G

## Používání Bash v Colabu

Jak bylo zmíněno, `!` se používá pro jednotlivé shell příkazy. Pro spuštění celých Bash skriptů v buňce můžete použít `%%bash`.

In [ ]:
# Příklad použití '!' pro jednotlivý příkaz
!echo "Toto je spuštěno jako shell příkaz z Python buňky."

A teď příklad buňky s `%%bash`:

In [ ]:
%%bash
# Toto je celá buňka, která se vykoná jako Bash skript.
echo "Jsem v Bash buňce."
CURRENT_DIR=$(pwd)
echo "Aktuální adresář je: $CURRENT_DIR"
ls -l

Jsem v Bash buňce.
Aktuální adresář je: /content
total 4
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


## Tips for other things to show

Run the commandline interface to Udapi, capture its HTML output, then either show it in a notebook cell or let the user download it.

In [5]:
from IPython.display import HTML
from google.colab import files

# Capture HTML output from a shell command in a variable.
###!!! Saving this as udapi_output.html will not work because it mixes STDOUT with STDERR, and the latter is not even in HTML.
#html_output = !cat UD_Czech-PUD/cs_pud-ud-test.conllu | udapy -HAM util.Mark node='node.upos == "PRON"'
!cat UD_Czech-PUD/cs_pud-ud-test.conllu | udapy -HAM util.Mark node='node.upos == "PRON"' > udapi_output.html

# html_output is a list of strings, we have to join them into one string.
#html_content = "\n".join(html_output)

# Show the HTML content in a notebook cell.
#HTML(html_content)

# Save the HTML content in a file in the virtual machine.
output_filename = 'udapi_output.html'
#with open(output_filename, 'w') as f:
#    f.write(html_content)

# Offer the file for download.
print(f"File '{output_filename}' is ready for download.")
files.download(output_filename)


2026-07-12 18:48:00,869 [   INFO] run_blocks - No reader specified, using read.Conllu
2026-07-12 18:48:00,869 [   INFO] run_blocks -  ---- ROUND ----
2026-07-12 18:48:00,869 [   INFO] run_blocks - Executing block read.Conllu {}
2026-07-12 18:48:00,870 [   INFO] try_fast_load - Reading -
2026-07-12 18:48:00,991 [   INFO] run_blocks - Executing block util.Mark node=node.upos == "PRON"
2026-07-12 18:48:01,198 [   INFO] run_blocks - Executing block write.TextModeTreesHtml color=1 attributes=form,lemma,upos,xpos,feats,deprel,misc marked_only=1
File 'udapi_output.html' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>